# Data 영역 코드블록 단위 빈칸채우기 문제 및 해설

AI 인증시험 출제포인트의 Data 항목과 시계열/추천 시스템 노트북을 기준으로, 원본 실습 코드블록 단위에 가깝게 문제를 다시 구성했습니다.

## 구성 기준

- 작은 문법 단위로 쪼개지 않고, 원본 노트북의 주요 코드 셀 흐름을 하나의 문제로 묶었습니다.
- 각 문제는 `문제 설명 -> 빈칸 코드블록 -> 채워야 할 내용 -> 해설` 순서입니다.
- Data 출제포인트와 관련된 시계열 모델, 추천 모델, 학습/평가/metric 흐름만 포함했습니다.

## 시계열 예측

### 문제 1. Training / Evaluation Loop

아래는 시계열 예측 모델의 학습 및 평가 루프입니다. device 이동, 마지막 timestep 예측 추출, loss 계산, gradient 초기화/역전파/업데이트, 평가 루프를 완성하세요.

In [ ]:
def train(model, train_loader, test_loader):
  train_hist = []
  test_hist = []
  num_epochs = 10
  loss_fn = nn.MSELoss(reduction='mean')
  optimizer = optim.Adam(model.parameters(), lr=1e-3)

  for epoch in range(num_epochs):
    total_train_loss = 0.0
    total_test_loss = 0.0

    # train
    model.train()
    for batch_x, batch_y in train_loader:
      batch_x, batch_y = ____
      pred = ____
      loss = ____

      optimizer.____
      loss.____
      optimizer.____

      total_train_loss += loss.item()

    avg_loss = total_train_loss / len(train_loader)
    train_hist.append(avg_loss)

    # evaluate
    model.eval()
    with torch.no_grad():
      for test_x, test_y in test_loader:
        test_x, test_y = ____
        test_pred = ____
        test_loss = ____
        total_test_loss += test_loss.item()

    avg_test_loss = total_test_loss / len(test_loader)
    test_hist.append(avg_test_loss)

    print(f'Epoch {epoch + 1:2d}/{num_epochs} - Training Loss: {avg_loss:.4f}, Test Loss: {avg_test_loss:.4f}')

  x = np.linspace(1, num_epochs, num_epochs)
  plt.plot(x, train_hist, scalex=True, label="Training loss")
  plt.plot(x, test_hist, label="Test loss")
  plt.legend()

**정답**

1. `batch_x.to(device), batch_y.to(device)`

   해설: 입력과 정답 텐서를 모델이 올라간 device로 옮겨야 같은 장치에서 연산할 수 있습니다.

2. `model(batch_x)[:, -1, 0]`

   해설: 모델 출력이 [batch, sequence, feature] 형태이므로 마지막 timestep의 첫 번째 feature를 예측값으로 사용합니다.

3. `loss_fn(pred, batch_y)`

   해설: 정의된 MSE loss 함수에 예측값과 실제 정답을 넣어 학습 손실을 계산합니다.

4. `zero_grad()`

   해설: PyTorch는 gradient가 누적되므로 매 batch마다 이전 gradient를 먼저 초기화합니다.

5. `backward()`

   해설: 현재 loss를 기준으로 각 파라미터의 gradient를 계산합니다.

6. `step()`

   해설: optimizer가 계산된 gradient를 사용해 모델 파라미터를 갱신합니다.

7. `test_x.to(device), test_y.to(device)`

   해설: 평가 시에도 입력과 정답을 모델과 같은 device로 옮겨야 합니다.

8. `model(test_x)[:, -1, 0]`

   해설: 테스트 출력에서도 학습 때와 동일하게 마지막 timestep 값을 예측 결과로 사용합니다.

9. `loss_fn(test_pred, test_y)`

   해설: 테스트 예측값과 정답 사이의 손실을 계산해 평가 loss를 누적합니다.

**해설**

학습 루프에서는 모델과 입력/정답 텐서를 같은 device에 올린 뒤, 모델 출력 중 마지막 timestep의 값을 예측값으로 사용합니다. loss.backward()로 gradient를 계산하고 optimizer.step()으로 파라미터를 갱신합니다. 평가 루프에서는 model.eval()과 torch.no_grad()를 사용해 추론 전용으로 loss를 계산합니다.

### 문제 2. Test Metric

테스트 데이터에서 예측값을 만든 뒤 RMSE와 MAPE를 계산하는 함수입니다. metric 함수의 입력 순서를 완성하세요.

In [ ]:
def test(model, X_test, y_test):
  model.eval()
  with torch.no_grad():
    X_test = X_test.to(device)
    y_hat = model(X_test)
    test_predictions = y_hat[:, -1, 0]

  test_predictions = test_predictions.cpu().numpy()
  y_test = y_test.cpu().numpy()

  rmse = root_mean_squared_error(____, ____)
  mape = mean_absolute_percentage_error(____, ____)

  return rmse, mape

**정답**

1. `y_test, test_predictions`

   해설: RMSE는 실제값을 먼저, 예측값을 나중에 넣어 오차의 크기를 계산합니다.

2. `y_test, test_predictions`

   해설: MAPE도 실제값 기준으로 예측값의 상대 오차를 계산하므로 같은 순서를 사용합니다.

**해설**

sklearn 계열 metric은 보통 정답(y_true)을 먼저, 예측값(y_pred)을 나중에 넣습니다. 여기서는 y_test가 실제값이고 test_predictions가 모델 예측값입니다.

### 문제 3. Conv1D Model

시계열 입력을 Conv1D로 처리하는 모델입니다. Conv1d의 채널 수와 Linear 입력 차원을 완성하세요.

In [ ]:
class Conv1DModel(nn.Module):
  def __init__(self, input_size, hidden_size):
    super(Conv1DModel, self).__init__()
    self.conv1d = nn.Conv1d(in_channels=____,
                            out_channels=____,
                            kernel_size=2, stride=1)
    self.fc = nn.Linear(____, 1)

  def forward(self, x):
    x = x.transpose(1, 2)
    x = self.conv1d(x)
    x = x.transpose(1, 2)
    return self.fc(x)

**정답**

1. `input_size`

   해설: Conv1d의 in_channels는 입력 feature 수와 일치해야 합니다.

2. `hidden_size`

   해설: Conv1d가 만들어낼 출력 채널 수를 hidden_size로 설정합니다.

3. `hidden_size`

   해설: Conv1d 출력의 feature 차원이 hidden_size이므로 Linear의 입력 차원도 hidden_size입니다.

**해설**

PyTorch Conv1d는 [batch, channel, sequence] 형식을 기대하므로 입력을 transpose합니다. 입력 feature 수가 in_channels이고, convolution 출력 채널 수가 hidden_size이므로 Linear의 입력 차원도 hidden_size입니다.

### 문제 4. RNN Model

기본 RNN 기반 시계열 예측 모델입니다. RNN 계층 정의와 forward 출력을 완성하세요.

In [ ]:
class RNNModel(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers):
      super(RNNModel, self).__init__()
      self.rnn = nn.RNN(____, ____, ____, batch_first=True)
      self.fc = nn.Linear(____, 1)

  def forward(self, x):
      ____
      ____

**정답**

1. `input_size, hidden_size, num_layers`

   해설: RNN 계층에는 입력 feature 차원, hidden 차원, layer 수가 순서대로 들어갑니다.

2. `hidden_size`

   해설: RNN 출력의 마지막 차원이 hidden_size이므로 Linear 입력 차원도 hidden_size입니다.

3. `out, _ = self.rnn(x)`

   해설: RNN은 전체 timestep 출력 out과 마지막 hidden state를 반환하며, 여기서는 out만 사용합니다.

4. `return self.fc(out)`

   해설: 각 timestep의 hidden representation을 FC 계층에 통과시켜 예측값 sequence를 만듭니다.

**해설**

RNN은 입력 차원, hidden 차원, layer 수를 받아 sequence를 처리합니다. batch_first=True이므로 입력은 [batch, sequence, feature] 형태이며, 각 timestep의 hidden 출력 out을 Linear 계층으로 예측값에 매핑합니다.

### 문제 5. Encoder-Decoder RNN

멀티스텝 예측용 Encoder-Decoder RNN입니다. Encoder의 hidden state 반환과 Decoder의 다음 값 생성 흐름을 완성하세요.

In [ ]:
class EncoderRNN(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers):
    super(EncoderRNN, self).__init__()
    self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

  def forward(self, x):
    ____
    ____

class DecoderRNN(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers):
    super(DecoderRNN, self).__init__()
    self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
    self.fc = nn.Linear(hidden_size, input_size)

  def forward(self, x, h):
    ____
    ____
    ____

class RNNRNN(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers):
    super(RNNRNN, self).__init__()
    self.encoder = EncoderRNN(input_size, hidden_size, num_layers)
    self.decoder = DecoderRNN(input_size, hidden_size, num_layers)

  def forward(self, source, target_len):
    h = self.encoder(source)
    predictions = []
    input = source[:, -1, :].unsqueeze(1)
    for t in range(target_len):
      out, h = self.decoder(input, h)
      predictions.append(out.squeeze(1))
      input = out
    outputs = torch.stack(predictions, dim=1)
    return outputs

**정답**

1. `_, h = self.rnn(x)`

   해설: Encoder는 전체 입력 sequence를 읽고 마지막 hidden state h를 얻습니다.

2. `return h`

   해설: Decoder의 초기 hidden state로 쓰기 위해 h만 반환합니다.

3. `out, h = self.rnn(x, h)`

   해설: Decoder는 현재 입력 x와 이전 hidden state h를 함께 받아 다음 출력을 계산합니다.

4. `out = self.fc(out)`

   해설: RNN hidden 출력을 실제 예측 feature 차원으로 변환합니다.

5. `return out, h`

   해설: 다음 timestep 예측값과 갱신된 hidden state를 함께 반환해야 반복 디코딩이 가능합니다.

**해설**

Encoder는 입력 sequence 전체를 읽고 마지막 hidden state h를 요약 표현으로 반환합니다. Decoder는 이전 입력 x와 hidden state h를 받아 다음 timestep 출력을 만들고, 그 출력을 다음 decoder 입력으로 다시 사용합니다.

## 추천 시스템

### 문제 6. NGCF Layer

NGCF의 message passing layer입니다. degree 기반 정규화, 사용자-아이템 메시지 계산, 아이템 노드 집계 및 self-connection을 완성하세요.

In [ ]:
class NGCFLayer(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.1):
        super().__init__()
        self.W1 = nn.Linear(input_dim, output_dim)
        self.W2 = nn.Linear(input_dim, output_dim)
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, edge_index, node_features, user_num, item_num):
        src, dst = edge_index

        deg = torch.zeros(node_features.size(0), device=node_features.device)
        deg.index_add_(0, src, torch.ones_like(src, dtype=torch.float))
        deg.index_add_(0, dst, torch.ones_like(dst, dtype=torch.float))

        norm = ____

        src_feat = node_features[src]
        dst_feat = node_features[dst]

        edge_messages_for_src = self.W1(dst_feat) + self.W2(src_feat * dst_feat)
        edge_messages_for_src *= norm.unsqueeze(1)

        edge_messages_for_dst = ____
        edge_messages_for_dst *= ____

        aggregated_messages = torch.zeros_like(node_features)

        aggregated_messages.index_add(0, src, edge_messages_for_src)
        aggregated_messages[:user_num] += self.W1(node_features[:user_num])

        ____
        ____

        aggregated_features = self.leaky_relu(aggregated_messages)
        return aggregated_features

**정답**

1. `1.0 / torch.sqrt(deg[src] * deg[dst])`

   해설: edge 양 끝 노드의 degree를 이용해 연결 수가 많은 노드의 영향력을 정규화합니다.

2. `self.W1(src_feat) + self.W2(src_feat * dst_feat)`

   해설: 아이템 노드가 사용자 노드로부터 받는 메시지는 이웃 표현과 사용자-아이템 상호작용 항을 함께 사용합니다.

3. `norm.unsqueeze(1)`

   해설: edge별 정규화 스칼라를 feature 차원에 broadcast하기 위해 차원을 하나 늘립니다.

4. `aggregated_messages.index_add_(0, dst, edge_messages_for_dst)`

   해설: 같은 아이템 노드로 들어오는 여러 사용자 메시지를 dst 인덱스 기준으로 누적합니다.

5. `aggregated_messages[user_num:] += self.W1(node_features[user_num:])`

   해설: 아이템 노드 자기 자신의 표현도 self-connection 메시지로 더합니다.

**해설**

NGCF는 사용자와 아이템을 bipartite graph의 노드로 보고 edge를 따라 메시지를 전달합니다. degree 정규화로 노드 연결 수 차이를 보정하고, W1 이웃 표현과 W2 상호작용 항을 합쳐 메시지를 만듭니다. 이후 같은 노드로 들어오는 메시지를 index_add로 집계하고 self-connection을 더합니다.

### 문제 7. NGCF Model

NGCF 전체 모델의 forward입니다. 초기 embedding에서 layer를 순차 적용하고, layer별 representation을 concat한 뒤 user/item feature로 분리하세요.

In [ ]:
class NGCF(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim, layer_dims, dropout=0.1):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim

        self.node_embeddings = nn.Embedding(self.num_users + self.num_items, self.embedding_dim)
        nn.init.xavier_uniform_(self.node_embeddings.weight)

        self.layers = nn.ModuleList([
            NGCFLayer(input_dim=(embedding_dim if i == 0 else layer_dims[i - 1]),
                      output_dim=layer_dims[i], dropout=dropout)
            for i in range(len(layer_dims))
        ])

    def forward(self, edge_index):
        node_features = self.node_embeddings.weight
        layer_outputs = [node_features]
        for layer in self.layers:
            node_features = ____
            layer_outputs.append(node_features)

        final_features = ____
        user_features = ____
        item_features = ____
        return user_features, item_features

**정답**

1. `layer(edge_index, node_features, self.num_users, self.num_items)`

   해설: 현재 node feature를 NGCF layer에 넣어 다음 layer의 representation을 계산합니다.

2. `torch.concat(layer_outputs, dim=-1)`

   해설: 초기 embedding과 각 layer 출력을 feature 차원으로 이어 붙여 최종 표현을 만듭니다.

3. `final_features[:self.num_users]`

   해설: 노드 배열의 앞부분은 사용자 노드이므로 사용자 feature로 분리합니다.

4. `final_features[self.num_users:]`

   해설: 사용자 노드 이후의 나머지 부분은 아이템 노드 feature입니다.

**해설**

NGCF는 초기 embedding뿐 아니라 각 graph convolution layer의 출력을 모두 활용합니다. concat된 final_features에서 앞쪽 num_users개는 사용자 표현, 그 뒤는 아이템 표현입니다.

### 문제 8. NCF Model / Training / Metrics

NCF의 embedding 결합, 학습 설정, Recall@K/Precision@K 계산 흐름입니다. 원본 코드 흐름에 맞게 빈칸을 완성하세요.

In [ ]:
class Neural_Collaborative_Filtering(nn.Module):
  def __init__(self, n_users, n_movies):
    super().__init__()
    self.user_embedding = nn.Embedding(n_users, 32)
    self.movie_embedding = nn.Embedding(n_movies, 32)

    self.fc1 = nn.Linear(64, 32)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(32, 1)

  def forward(self, users, movies, ratings=None):
    user_embedding = self.user_embedding(users)
    movie_embedding = self.movie_embedding(movies)
    input_embedding = ____

    hidden_feature = self.fc1(input_embedding)
    hidden_feature = self.relu(hidden_feature)
    output = self.fc2(hidden_feature)
    return output

optimizer = ____
loss_func = ____

# Recall@K / Precision@K
k = 10
threshold = 3.5

for user_id, user_ratings in user_est_true.items():
  user_ratings.sort(key=lambda x: x[0], reverse=True)
  n_real_relevant = sum((true_r >= threshold) for (_, true_r) in user_ratings)
  recommended_k = user_ratings[:k]
  n_real_relevant_in_top_k = sum((true_r >= threshold) for (est, true_r) in recommended_k)

  precisions[user_id] = n_real_relevant_in_top_k / ____
  if n_real_relevant:
    recalls[user_id] = n_real_relevant_in_top_k / ____
  else:
    recalls[user_id] = 0

**정답**

1. `torch.cat([user_embedding, movie_embedding], dim=1)`

   해설: NCF는 사용자 embedding과 아이템 embedding을 결합한 벡터를 MLP 입력으로 사용합니다.

2. `torch.optim.Adam(model.parameters(), lr=0.001)`

   해설: Adam optimizer가 embedding과 MLP 파라미터를 함께 업데이트하도록 설정합니다.

3. `nn.MSELoss(reduction='mean')`

   해설: 평점 예측은 회귀 문제이므로 예측 평점과 실제 평점의 평균제곱오차를 사용합니다.

4. `k`

   해설: Precision@K는 추천한 K개 중 실제 relevant item의 비율이므로 분모가 K입니다.

5. `n_real_relevant`

   해설: Recall@K는 전체 relevant item 중 top-K에 포함된 비율이므로 분모가 실제 relevant item 수입니다.

**해설**

NCF는 사용자 embedding과 아이템 embedding을 concat해 MLP 입력으로 사용합니다. 평점 예측은 회귀 문제이므로 MSELoss를 쓰고, optimizer는 모델 파라미터를 갱신합니다. Precision@K는 추천한 K개 중 relevant 비율, Recall@K는 실제 relevant item 중 top-K에 포함된 비율입니다.